# Milestone 2: IV measures

This notebook imports `earnings_vol` and defines no logic of its own. The walkthrough is `docs/milestones/02-iv-measures.md`; the decisions are ADR 0003.

To rebuild the tables from the option-chain extracts (it needs the local cache):

```
uv run python -m earnings_vol.events.run
uv run python -m earnings_vol.measures.run --only snapshots
uv run python -m earnings_vol.measures.run --only iv
uv run python tools/export_app_data.py
```

Contract schema version: `0.4.0`.

## 1. Black-76 on the parity forward (ADR 0003 §0 step 3)
The price depends on sigma and T only through the total variance w = sigma^2 T, so a price pins down w and no day count enters. A made-up call: F = 100, K = 110, w = 0.04, DF = 0.99.

In [ ]:
from earnings_vol.measures import black

price = float(black.black76(100.0, 110.0, 0.04, 0.99, True))
print(f"call {price:.6f}")
print(f"w back from the price {black.implied_w(price, 100.0, 110.0, 0.99, True):.10f}")
print("a call above Black76(w = 4) has no root:", black.implied_w(70.0, 100.0, 100.0, 1.0, True))


## 2. The clock: NYSE sessions to expiry (ADR 0003 §0 step 5)
Sessions in (snapshot, expiry]: the expiry counts, the snapshot session does not. Thanksgiving 2025 is closed and the half day after it counts as 1.

In [ ]:
from datetime import date

from earnings_vol.events import calendar
from earnings_vol.measures import chain

eras = {"weekly_from": "2019-02-09", "mwf_from": "2020-01-06",
        "daily_from": "2024-09-11", "data_end": "2026-10-26"}
sessions = calendar.Sessions(calendar.build_session_table(date(2025, 1, 1), date(2025, 12, 31), eras)
                             ["session_date"].tolist())
for exp in (date(2025, 11, 14), date(2025, 11, 28)):
    print("2025-10-30 ->", exp, chain.tau_sessions(date(2025, 10, 30), exp, sessions), "sessions,",
          (exp - date(2025, 10, 30)).days, "calendar days")


## 3. Event variance and the implied move (ADR 0003 §c)
The AAPL worked example's two ATM total variances on day -1 (2025-10-30), both expiries spanning the 2025-10-31 day 0: w1 = 0.0034862510 at 11 sessions, w2 = 0.0056163135 at 20. E = (w1 tau2 - w2 tau1) / (tau2 - tau1).

In [ ]:
from earnings_vol.measures import termstructure

e, s2 = termstructure.event_variance(0.0034862510, 11, 0.0056163135, 20)
moves = termstructure.implied_moves(e, s2)
print(f"E = {e:.8f}, sigma_d^2 = {s2:.4e} per session")
print(f"implied move sqrt(E) = {moves['implied_move']:.4%}, "
      f"expected |r0| = {moves['implied_abs_move_total']:.4%}")
print(f"calendar clock instead: E = {termstructure.event_variance(0.0034862510, 15, 0.0056163135, 29)[0]:.8f}")


## 4. The event-aware 30-day IV (ADR 0003 §b)
Made-up two-component variances, sigma_d^2 = 0.0002 per session and E = 0.002, 25 sessions before day 0: clean expiries at 10 and 19 sessions, a spanning one at 30. The event-aware IV uses the clean class only; the blind one interpolates across the event and leaks E * 2/11 into the horizon.

In [ ]:
xs = [termstructure.Expiry(date(2026, 2, 13), 10, 0.0020, chain.annualised_iv(0.0020, 10), False),
      termstructure.Expiry(date(2026, 2, 27), 19, 0.0038, chain.annualised_iv(0.0038, 19), False),
      termstructure.Expiry(date(2026, 3, 13), 30, 0.0080, chain.annualised_iv(0.0080, 30), True)]
day, _ = termstructure.day_measures(xs, sessions_to_event=25)
print(day["cm30_method"], f"cm30 {day['cm30_iv']:.4f}, blind {day['cm30_blind_iv']:.4f}")


## 5. The committed measures
`data/derived/event_iv.parquet` holds one row per event (E, the implied move, the crushes, the realised move); `iv_daily.parquet` one row per company and session. The AAPL event of the worked example, read with the pipeline's DTB3 rate and its full chain.

In [ ]:
from earnings_vol import params, schema, store

derived = params.ROOT / "data" / "derived"
eiv = store.read_table(derived / "event_iv.parquet", schema.EVENT_IV).set_index("event_id")
r = eiv.loc["AAPL:2025-10-30"]
print(f"E {r.evar_m1:.8f}, sqrt(E) {r.implied_move_m1:.4%}, cm30 crush {r.crush_mean_adj:.4f}, "
      f"front crush {r.crush_front_fixed:.4f}, realised {r.realized_ret_0:.4%}")
print(eiv["implied_move_status"].value_counts().to_dict())
